In [2]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sqlalchemy import create_engine
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [3]:
# Download the VADER lexicon for sentiment analysis if not already present.
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\karim\AppData\Roaming\nltk_data...


True

In [18]:
# CREATE SQLALCHEMY DATABASE CONNECTION

server = r"localhost\SQLEXPRESS"
database = "MarketingAnalytics"
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    f"?driver={driver.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

In [19]:
# FETCH CUSTOMER REVIEWS FROM SQL SERVER

query = """
SELECT 
    ReviewID,
    CustomerID,
    ProductID,
    ReviewDate,
    Rating,
    ReviewText
FROM customer_reviews
"""

customer_reviews_df = pd.read_sql(query, engine)

print("Data loaded successfully!")
print(customer_reviews_df.head())

Data loaded successfully!
   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  
0   Average  experience,  nothing  special.  
1            The  quality  is    top-notch.  
2   Five  stars  for  the  quick  delivery.  
3  Good  quality,  but  could  be  cheaper.  
4   Average  experience,  nothing  special.  


In [20]:
# INITIALIZE VADER
sia = SentimentIntensityAnalyzer()

In [21]:
# CALCULATE SENTIMENT SCORE

def calculate_sentiment(review):
    if pd.isna(review):
        return 0.0

    sentiment = sia.polarity_scores(str(review))

    return sentiment["compound"]


customer_reviews_df["SentimentScore"] = (
    customer_reviews_df["ReviewText"]
    .apply(calculate_sentiment)
)

In [22]:
# CATEGORIZE SENTIMENT

def categorize_sentiment(score, rating):

    if score > 0.05:

        if rating >= 4:
            return "Positive"

        elif rating == 3:
            return "Mixed Positive"

        else:
            return "Mixed Negative"

    elif score < -0.05:

        if rating <= 2:
            return "Negative"

        elif rating == 3:
            return "Mixed Negative"

        else:
            return "Mixed Positive"

    else:

        if rating >= 4:
            return "Positive"

        elif rating <= 2:
            return "Negative"

        else:
            return "Neutral"


customer_reviews_df["SentimentCategory"] = customer_reviews_df.apply(
    lambda row: categorize_sentiment(
        row["SentimentScore"],
        row["Rating"]
    ),
    axis=1
)

In [23]:
# 6. CREATE SENTIMENT BUCKETS

def sentiment_bucket(score):

    if score >= 0.5:
        return "0.5 to 1.0"

    elif 0.0 <= score < 0.5:
        return "0.0 to 0.49"

    elif -0.5 <= score < 0.0:
        return "-0.49 to 0.0"

    else:
        return "-1.0 to -0.5"


customer_reviews_df["SentimentBucket"] = (
    customer_reviews_df["SentimentScore"]
    .apply(sentiment_bucket)
)


In [24]:
# DISPLAY RESULTS

print("\nSentiment Analysis Results:")
print(customer_reviews_df.head())



Sentiment Analysis Results:
   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089    Mixed Negative   
1            The  quality  is    top-notch.          0.0000          Positive   
2   Five  stars  for  the  quick  delivery.          0.0000          Positive   
3  Good  quality,  but  could  be  cheaper.          0.2382    Mixed Positive   
4   Average  experience,  nothing  special.         -0.3089    Mixed Negative   

  SentimentBucket  
0    -0.49 to 0.0  
1     0.0 to 0.49  
2     0.0 to 0.49  
3     0.0 to 0.49  
4    -0.49 to 0.0  


In [25]:
# SAVE RESULTS TO CSV

customer_reviews_df.to_csv(
    "customer_reviews_with_sentiment.csv",
    index=False
)

print("\nCSV file created successfully!")


CSV file created successfully!
